# অধ্যায় ৪: ফিচার ইঞ্জিনিয়ারিং
## পাঠ ৪.৩: ফিচার সিলেকশন

আজ আমরা শিখব কীভাবে অপ্রয়োজনীয় ফিচার বাদ দিয়ে আমাদের মডেলকে উন্নত করা যায়।

### A. কেন সব ফিচার দরকারি নয়?

তুমি একটি বড় মার্কেটে দাঁড়িয়ে আছ। তুমি তোমার বন্ধুকে খুঁজছ। তোমার কাছে অনেক তথ্য আছে:
- বন্ধুর নাম
- বন্ধুর বয়স
- বন্ধুর উচ্চতা
- মার্কেটের তাপমাত্রা
- আজকের দিনের নাম
- মার্কেটের মেঝের রং

এর মধ্যে কোন তথ্যগুলো তোমার বন্ধুকে খুঁজতে সাহায্য করবে? নাম, বয়স, উচ্চতা—হ্যাঁ। কিন্তু তাপমাত্রা, দিনের নাম, মেঝের রং—না। এই অপ্রয়োজনীয় তথ্যগুলোকে 'নয়জ' (noise) বলে। মেশিন লার্নিং-এ নয়জ মডেলকে বিভ্রান্ত করে এবং ওভারফিটিং তৈরি করতে পারে।

### B. ফিচার সিলেকশন কেন গুরুত্বপূর্ণ?

১. **গতি**: কম ফিচার = দ্রুত ট্রেনিং
২. **সরলতা**: সহজ মডেল বোঝা এবং ব্যাখ্যা করা সহজ
৩. **ওভারফিটিং কমানো**: অপ্রাসঙ্গিক ফিচার বাদ দিলে মডেল জেনারেলাইজ ভালো করে
৪. **কার্যক্ষমতা**: কিছু ফিচার মডেলের কার্যক্ষমতা কমিয়ে দেয়

### C. ফিচার সিলেকশনের তিনটি প্রধান পদ্ধতি

**১. ফিল্টার পদ্ধতি (Filter Method):** পরিসংখ্যানিক টেস্ট ব্যবহার করে ফিচার বাছাই। যেমন SelectKBest।
**২. র্যাপার পদ্ধতি (Wrapper Method):** বিভিন্ন ফিচার কম্বিনেশন ট্রাই করে সেরাটি বাছাই। যেমন RFE।
**৩. এম্বেডেড পদ্ধতি (Embedded Method):** মডেল নিজেই ফিচারের গুরুত্ব নির্ধারণ করে। যেমন SelectFromModel।

এখন আমরা প্রতিটি পদ্ধতি কোড সহ দেখব।

In [1]:
# প্রয়োজনীয় লাইব্রেরি
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# একটি ডেটাসেট তৈরি করি যেখানে অনেক অপ্রয়োজনীয় ফিচার আছে
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=5, 
    n_redundant=5, n_repeated=5, random_state=42
)
print('ডেটাসেটের আকৃতি:', X.shape)
print('মোট ফিচার:', X.shape[1])
print('শুধু', 5, 'টি ফিচার সত্যিই তথ্যবহুল!')

ডেটাসেটের আকৃতি: (1000, 20)
মোট ফিচার: 20
শুধু 5 টি ফিচার সত্যিই তথ্যবহুল!


### D. পদ্ধতি ১: SelectKBest (ফিল্টার পদ্ধতি)

এই পদ্ধতিতে আমরা একটি পরিসংখ্যানিক টেস্ট (যেমন f_classif) ব্যবহার করে প্রতিটি ফিচারের স্কোর বের করি এবং সেরা Kটি ফিচার নির্বাচন করি।

f_classif কিভাবে কাজ করে? এটি ANOVA F-টেস্ট ব্যবহার করে। সহজ ভাষায়, এটি দেখে প্রতিটি ফিচারের মান বিভিন্ন ক্লাসে কতটা আলাদা। যদি একটি ফিচারের মান ক্লাস ০ এবং ক্লাস ১-এ খুব আলাদা হয়, তার স্কোর বেশি হবে।

In [2]:
# SelectKBest ব্যবহার করি
selector_kbest = SelectKBest(score_func=f_classif, k=5)
X_kbest = selector_kbest.fit_transform(X, y)

# প্রতিটি ফিচারের স্কোর দেখি
scores = selector_kbest.scores_
top_indices = np.argsort(scores)[::-1][:10]
print('সেরা 10 ফিচারের স্কোর:')
for i, idx in enumerate(top_indices):
    print(f'  ফিচার {idx:2d}: {scores[idx]:.2f}')
print(f'\nনির্বাচিত ফিচার (শেষ): {selector_kbest.get_support().sum()} টি')
print('নির্বাচিত ফিচার ইন্ডেক্স:', np.where(selector_kbest.get_support())[0])

সেরা 10 ফিচারের স্কোর:
  ফিচার  8: 684.06
  ফিচার 11: 132.72
  ফিচার  3: 132.72
  ফিচার 18: 114.90
  ফিচার 13: 114.90
  ফিচার 10: 108.02
  ফিচার 14: 95.76
  ফিচার  1: 56.29
  ফিচার  5: 44.27
  ফিচার 12: 44.27

নির্বাচিত ফিচার (শেষ): 5 টি
নির্বাচিত ফিচার ইন্ডেক্স: [ 3  8 11 13 18]


### E. পদ্ধতি ২: SelectFromModel (এম্বেডেড পদ্ধতি)

এখানে আমরা একটি মডেল ট্রেইন করি এবং মডেলটি নিজেই বলে দেয় কোন ফিচারগুলো গুরুত্বপূর্ণ। Random Forest-এর `feature_importances_` বা Logistic Regression-এর `coef_` ব্যবহার করে আমরা ফিচার সিলেক্ট করতে পারি।

**কৌশল:** `threshold='median'` বা `threshold='mean'` দিয়ে আমরা বলি—মাঝামাঝি বা গড় গুরুত্বের চেয়ে কম গুরুত্বপূর্ণ ফিচারগুলো বাদ দাও।

In [3]:
# SelectFromModel ব্যবহার করি
rf = RandomForestClassifier(n_estimators=100, random_state=42)
selector_model = SelectFromModel(rf, threshold='median')
X_model = selector_model.fit_transform(X, y)

# ফিচারের গুরুত্ব দেখি
importances = selector_model.estimator_.feature_importances_
top_idx = np.argsort(importances)[::-1][:10]
print('Random Forest অনুযায়ী সেরা 10 ফিচারের গুরুত্ব:')
for i, idx in enumerate(top_idx):
    print(f'  ফিচার {idx:2d}: {importances[idx]:.4f}')
print(f'\nSelectFromModel নির্বাচিত করেছে: {X_model.shape[1]} টি ফিচার')

Random Forest অনুযায়ী সেরা 10 ফিচারের গুরুত্ব:
  ফিচার  8: 0.1881
  ফিচার 10: 0.1024
  ফিচার 15: 0.0953
  ফিচার 17: 0.0928
  ফিচার  1: 0.0855
  ফিচার 14: 0.0604
  ফিচার 18: 0.0503
  ফিচার 16: 0.0450
  ফিচার 13: 0.0415
  ফিচার  3: 0.0414

SelectFromModel নির্বাচিত করেছে: 10 টি ফিচার


### F. পদ্ধতি ৩: RFE (রিকার্সিভ ফিচার এলিমিনেশন - র্যাপার পদ্ধতি)

RFE-র কাজের পদ্ধতি খুব মজার। এটি প্রথমে সব ফিচার দিয়ে মডেল ট্রেইন করে, তারপর সবচেয়ে কম গুরুত্বপূর্ণ ফিচারটি বাদ দেয়। তারপর আবার ট্রেইন করে, আবার বাদ দেয়—যতক্ষণ না নির্দিষ্ট সংখ্যক ফিচার থাকে। অর্থাৎ, এটি বারবার ফিচার বাদ দিয়ে মডেলের কর্মক্ষমতা পর্যবেক্ষণ করে।

**সতর্কতা:** RFE ধীর হতে পারে কারণ এটি প্রতিটি স্টেপে মডেল পুনরায় ট্রেইন করে। কিন্তু ফলাফল সাধারণত ভালো হয়।

In [4]:
# RFE ব্যবহার করি
lr = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr, n_features_to_select=5)
X_rfe = rfe.fit_transform(X, y)
print('RFE নির্বাচিত ফিচার:', rfe.get_support().sum(), 'টি')
print('নির্বাচিত ফিচার ইন্ডেক্স:', np.where(rfe.get_support())[0])
print('\nফিচার র‍্যাঙ্কিং:', rfe.ranking_)

RFE নির্বাচিত ফিচার: 5 টি
নির্বাচিত ফিচার ইন্ডেক্স: [ 1  8 11 14 16]

ফিচার র‍্যাঙ্কিং: [14  1  3  4  5 10 11  6  1 12  2  1  8  9  1 16  1 15  7 13]


### G. তিনটি পদ্ধতির তুলনা

এখন আমরা দেখব ফিচার সিলেকশন আসলে মডেলের কর্মক্ষমতা উন্নত করে কিনা। আমরা তিনটি পদ্ধতি ট্রাই করব এবং Accuracy তুলনা করব।

In [5]:
# ডেটা ভাগ করি
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. সব ফিচার দিয়ে
rf_all = RandomForestClassifier(n_estimators=100, random_state=42)
rf_all.fit(X_train, y_train)
acc_all = accuracy_score(y_test, rf_all.predict(X_test))

# 2. SelectKBest দিয়ে
selector = SelectKBest(f_classif, k=5)
X_train_kbest = selector.fit_transform(X_train, y_train)
X_test_kbest = selector.transform(X_test)
rf_kbest = RandomForestClassifier(n_estimators=100, random_state=42)
rf_kbest.fit(X_train_kbest, y_train)
acc_kbest = accuracy_score(y_test, rf_kbest.predict(X_test_kbest))

# 3. SelectFromModel দিয়ে
selector_sfm = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    threshold='median'
)
X_train_sfm = selector_sfm.fit_transform(X_train, y_train)
X_test_sfm = selector_sfm.transform(X_test)
rf_sfm = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sfm.fit(X_train_sfm, y_train)
acc_sfm = accuracy_score(y_test, rf_sfm.predict(X_test_sfm))

# 4. RFE দিয়ে
rfe_sel = RFE(LogisticRegression(max_iter=1000), n_features_to_select=5)
X_train_rfe = rfe_sel.fit_transform(X_train, y_train)
X_test_rfe = rfe_sel.transform(X_test)
rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rfe.fit(X_train_rfe, y_train)
acc_rfe = accuracy_score(y_test, rf_rfe.predict(X_test_rfe))
print('ফিচার সিলেকশন পদ্ধতির তুলনা:')
print(f'  1. সব ফিচার ({X.shape[1]}টি):         Accuracy = {acc_all:.4f}')
print(f'  2. SelectKBest (5টি):                 Accuracy = {acc_kbest:.4f}')
print(f'  3. SelectFromModel ({X_train_sfm.shape[1]}টি):        Accuracy = {acc_sfm:.4f}')
print(f'  4. RFE (5টি):                        Accuracy = {acc_rfe:.4f}')
print('\n→ দেখো, কম ফিচার দিয়েও অনেক সময় ভালো Accuracy পাওয়া যায়!')

ফিচার সিলেকশন পদ্ধতির তুলনা:
  1. সব ফিচার (20টি):         Accuracy = 0.9233
  2. SelectKBest (5টি):                 Accuracy = 0.8967
  3. SelectFromModel (10টি):        Accuracy = 0.9067
  4. RFE (5টি):                        Accuracy = 0.9033

→ দেখো, কম ফিচার দিয়েও অনেক সময় ভালো Accuracy পাওয়া যায়!


### H. আসল ডেটাসেটে উদাহরণ: Iris

এখন আমরা Iris ডেটাসেটে ফিচার সিলেকশন প্রয়োগ করব। Iris-এ ৪টি ফিচার আছে—দেখি সবকটি কি সত্যিই দরকারি।

In [6]:
# Iris ডেটাসেট
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
feature_names = iris.feature_names
print('Iris ফিচার:', feature_names)

# প্রতিটি ফিচারের জন্য f_classif স্কোর
selector_iris = SelectKBest(f_classif, k='all')
selector_iris.fit(X_iris, y_iris)

for name, score in zip(feature_names, selector_iris.scores_):
    print(f'  {name}: {score:.1f}')

# সেরা 2টি ফিচার বাছাই
selector_iris = SelectKBest(f_classif, k=2)
X_iris_sel = selector_iris.fit_transform(X_iris, y_iris)
selected = [feature_names[i] for i in np.where(selector_iris.get_support())[0]]
print(f'\nসেরা 2 ফিচার: {selected}')

Iris ফিচার: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
  sepal length (cm): 119.3
  sepal width (cm): 49.2
  petal length (cm): 1180.2
  petal width (cm): 960.0

সেরা 2 ফিচার: ['petal length (cm)', 'petal width (cm)']


### I. তুমি কি বুঝতে পেরেছ?

**প্রশ্ন ১:** ফিচার সিলেকশন কেন গুরুত্বপূর্ণ?

**প্রশ্ন ২:** SelectKBest, SelectFromModel এবং RFE-র মধ্যে মৌলিক পার্থক্য কী?

**প্রশ্ন ৩:** ফিল্টার পদ্ধতি (SelectKBest) এবং র্যাপার পদ্ধতি (RFE)-এর মধ্যে কোনটি ধীর? কেন?

**প্রশ্ন ৪:** সব ফিচার ব্যবহার করার চেয়ে কম ফিচার দিয়ে মাঝে মাঝে কেন ভালো ফল পাওয়া যায়?

### J. সারসংক্ষেপ

আজ আমরা শিখলাম:
✅ সব ফিচার দরকারি নয়—অপ্রয়োজনীয় ফিচার মডেলের ক্ষতি করতে পারে
✅ ফিচার সিলেকশনের তিনটি পদ্ধতি: ফিল্টার, র্যাপার, এম্বেডেড
✅ SelectKBest পরিসংখ্যানিক টেস্ট ব্যবহার করে ফিচার বাছাই করে
✅ SelectFromModel মডেলের ফিচার ইম্পরট্যান্স ব্যবহার করে
✅ RFE বারবার মডেল ট্রেইন করে সবচেয়ে দুর্বল ফিচার বাদ দেয়
✅ কম ফিচার দিয়েও অনেক সময় ভালো Accuracy পাওয়া যায়

এভাবে আমরা অধ্যায় ৪ শেষ করলাম। পরবর্তী অধ্যায়ে আমরা মডেল মূল্যায়ন নিয়ে শিখব!